In [ ]:
import glob
import os
import shutil
import zipfile

import polars as pl
import xarray as xr

# Preprocessing
In this step we will clean up the data collected in the previous notebook and transform it where needed.

## ENTSO-E

In [ ]:
INPUT_DIR = 'data/raw/entsoe'
OUTPUT_PATH = 'data/processed/entsoe_energy.parquet'

BIDDING_ZONES = ['DE_LU', 'DK1', 'DK2']

def load_zone(zone: str) -> pl.DataFrame:
    if zone == 'DK1': zone = '10YDK-1--------W'
    if zone == 'DK2': zone = '10YDK-2--------M'
    price_pattern = os.path.join(INPUT_DIR, f'{zone}_Price_*.parquet')
    price_parquets = sorted(glob.glob(price_pattern))
    price_df = pl.concat([pl.scan_parquet(f) for f in price_parquets])
    load_pattern = os.path.join(INPUT_DIR, f'{zone}_Load_*.parquet')
    load_parquets = sorted(glob.glob(load_pattern))
    load_df = pl.concat([pl.scan_parquet(f) for f in load_parquets])
    generation_pattern = os.path.join(INPUT_DIR, f'{zone}_Generation_*.parquet')
    generation_parquets = sorted(glob.glob(generation_pattern))
    generation_df = pl.concat([pl.scan_parquet(f) for f in generation_parquets], how="diagonal")

    price_df = price_df.interpolate() # fill holes in our data with linear interpolation
    load_df = load_df.interpolate()  # wether this is suitable for this kind of dataset is debatable
                           # however this does not actually occur in our datasets, since there are no whole in this data

    generation_df = generation_df.fill_null(0) # fill empty data in generation mix with zeroes.
    df = (
        pl.concat([price_df, load_df, generation_df], how='align')
          .rename({'__index_level_0__': 'time'})
    )
    
    df = df.collect()
    df = df.with_columns(pl.col('time').dt.convert_time_zone('Europe/Berlin'))
    df = (
        df.sort(by='time')
        .upsample(time_column='time', every='15m')
        .fill_null(strategy='forward') # upsampling, since we sometimes have the data only per hour.
    )
    return df


frames = [
    load_zone(z).select(
        'time',
        pl.all().exclude('time').name.prefix(f'{z}_')
        ) 
    for z in BIDDING_ZONES]


df = pl.concat(frames, how='align')
df.write_parquet(OUTPUT_PATH)
print(f'saved: {OUTPUT_PATH}  ({len(df)} lines)')

saved: data/processed/entsoe_energy.parquet  (245377 lines)


## Copernicus
For the weather data we have a bit more to do.
  - convert the netcdf files (or archives of netcdf files) to pandas dataframes
  - We have to combine the monthly data for each variable into one dataframe per country, like we already have for the energy data.
  - We have the wind speed given as u and v components, which we need to transform to one single wind speed
  - We have one data point per 0.25° by 0.25° sector for each of the variables which we will take a mean of.
  - The solar radiation is given as the prefix sum of the sequence, which we will differentiate back into the sequence.

In [66]:

INPUT_DIR = 'data/raw/era5'
OUTPUT_PATH = 'data/processed/era5_weather.parquet'

COUNTRIES = ['germany', 'luxembourg', 'denmark']


def ensure_unzipped(path: str) -> list[str]:
    if not zipfile.is_zipfile(path):
        return [path]
 
    extract_dir = path + '_unzipped'
    if not os.path.isdir(extract_dir):
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(path) as zf:
            zf.extractall(extract_dir)

    nc_files = sorted(glob.glob(os.path.join(extract_dir, '*.nc')))
    if not nc_files:
        raise ValueError(f'ZIP {path} does not contain .nc-files')
    return nc_files


def stream_group(nc_path: str) -> str:
    '''group data of the same type across months'''
    name = os.path.basename(nc_path).lower()
    if 'accum' in name:
        return 'accum'
    if 'instant' in name:
        return 'instant'
    return 'default'


def load_country(country: str) -> xr.Dataset:
    pattern = os.path.join(INPUT_DIR, f'era5_{country}_*.nc')
    zip_or_nc_files = sorted(glob.glob(pattern))
    if not zip_or_nc_files:
        raise FileNotFoundError(f'No file found for pattern: {pattern}')
 
    all_nc_files: list[str] = []
    for f in zip_or_nc_files:
        all_nc_files.extend(ensure_unzipped(f))
 
    groups: dict[str, list[str]] = {}
    for nc_path in all_nc_files:
        groups.setdefault(stream_group(nc_path), []).append(nc_path)
 
    group_datasets = [
        xr.open_mfdataset(sorted(paths), combine='by_coords')
        for paths in groups.values()
    ]
    ds = xr.merge(group_datasets, compat='override', join='inner')
 
    if 'valid_time' in ds.coords and 'time' not in ds.coords:
        ds = ds.rename({'valid_time': 'time'})
 
    return ds


def area_mean(ds: xr.Dataset) -> xr.Dataset:
    return ds.mean(dim=['latitude', 'longitude'], skipna=True) # Although the more southern latitudes will have slightly larger sectors,
                                                               # we just take the mean directly. The countries are span a bit more than 10°
                                                               # so this simplification shouldn't be a big problem.
                                                               # We also consider values for sectors, which aren't even part of the countries area,
                                                               # as we take the mean across the whole bounding box.
                                                               # This isn't clean but shouldn't be a huge issue.


def deaccumulate_ssrd(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        pl.when(pl.col("time").dt.hour() == 0)
        .then(pl.col("ssrd"))
        .otherwise(pl.col("ssrd").diff().over(pl.col("time").dt.truncate("1d")))
        .clip(lower_bound=0)
        .alias("solar_radiation_Jm2")
    )

def cleanup_unzipped() -> None:
    pattern = os.path.join(INPUT_DIR, '*_unzipped')
    for d in glob.glob(pattern):
        shutil.rmtree(d)

def process_country(country: str) -> pl.DataFrame:
    ds = load_country(country)
    
    ds = ds.copy()
    ds['wind_speed_10m'] = np.sqrt(ds['u10'] ** 2 + ds['v10'] ** 2)    # As this is a non-linear operation we have to perform it before taking
    ds['wind_speed_100m'] = np.sqrt(ds['u100'] ** 2 + ds['v100'] ** 2) # the mean otherwise opposing winds within a country would cancel out.

    ds = area_mean(ds)

    df = pl.DataFrame({
        'time': ds['time'].values,
        f'wind_speed_10m': ds['wind_speed_10m'].values,
        f'wind_speed_100m': ds['wind_speed_100m'].values,
        f'temperature_2m': ds['t2m'].values,
        f'ssrd': ds['ssrd'].values,
    })
    df.select(pl.col('temperature_2m') - 273.15)  # Kelvin -> Celsius

    df = df.sort('time')
    df = deaccumulate_ssrd(df)

    cleanup_unzipped()
    return df.select(
        'time',
        pl.all().exclude('time').name.prefix(f'{country}_')
    ) 


os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df = pl.concat([process_country(c) for c in COUNTRIES], how='align')

df.write_parquet(OUTPUT_PATH)
print(f'saved: {OUTPUT_PATH}  ({len(df)} lines)')


saved: data/processed/era5_weather.parquet  (61368 lines)


## Holidays

Since the demand for energy likely also coincides with weekdays and holidays, as the industrial energy usage is less of a factor on sundays and holidays, we also need this data. There is a holidays.csv file in data/public which we will use for this. Let's convert it to a parquet to stick with our format.

In [77]:
INPUT_PATH = 'data/public/holidays.csv'
OUTPUT_PATH = 'data/processed/holidays.parquet'

df = pl.read_csv(INPUT_PATH, schema_overrides={'weekday': pl.datatypes.Categorical})
df.write_parquet(OUTPUT_PATH)
print(f'saved: {OUTPUT_PATH}  ({len(df)} lines)')

saved: data/processed/holidays.parquet  (2557 lines)
